# How far does one machine go?

No Spark in this notebook. One process, one machine, and a table that keeps growing.

The other notebooks compare two engines at a fixed size. This one asks a different question: at
what point does a single process stop coping? It runs the same handful of queries over a table
that grows by a factor of 500, and records how long each takes.

Two things are worth watching more than the raw times.

**Throughput.** Rows processed per second. If it stays roughly flat as the table grows, the
engine is streaming rather than struggling — the work per row is constant and the total simply
scales with the row count.

**The memory cap.** It is held at a fixed, small value for every size below. By the last step the
file on disk is many times that cap. Nothing here loads the table into memory; if it did, the
run would fail long before the end.

Each size is generated, measured, then **deleted** before the next one starts, so peak disk usage
is one dataset rather than all of them. That is what makes the large sizes reachable at all.

In [ ]:
import sys, os, time, shutil
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C
import duckdb, pandas as pd
import matplotlib.pyplot as plt

# Held fixed while the data grows, on purpose. This is the number to watch.
SCALE_MEMORY_MB = C._env_int("BENCH_SCALE_MEM_MB", 1024)

LADDER = [1_000_000, 5_000_000, 25_000_000, 100_000_000, 250_000_000, 500_000_000]
if C.QUICK:
    LADDER = [500_000, 2_000_000, 10_000_000]

# Override to push further on a machine with the disk for it, e.g. BENCH_SCALE_MAX=2000000000
MAX_ROWS = C._env_int("BENCH_SCALE_MAX", LADDER[-1])
LADDER = [n for n in LADDER if n <= MAX_ROWS]

BYTES_PER_ROW = 17          # measured for this 7-column layout
SPILL_FACTOR  = 1.4         # room for the data plus what the engine spills
DISK_HEADROOM = 0.85        # never fill more than this share of free disk
STEP_TIME_BUDGET = C._env_int("BENCH_SCALE_STEP_SECONDS", 900)

SCALE_DIR = os.path.join(C.DATA_DIR, "scale")
os.makedirs(SCALE_DIR, exist_ok=True)

free_gb = shutil.disk_usage(SCALE_DIR).free / 1024**3
print(f"Machine     : {C.plural(C.CORES, 'thread')}, {C.RAM_GB:.1f} GB RAM, {free_gb:.1f} GB free disk")
print(f"Memory cap  : {SCALE_MEMORY_MB} MB, held fixed at every size")
print(f"Ladder      : {[C.human(n) for n in LADDER]}")
print(f"Largest step needs about "
      f"{LADDER[-1]*BYTES_PER_ROW*SPILL_FACTOR/1024**3:.1f} GB including spill "
      f"(deleted before the next one starts)")

## The queries

Four shapes, each doing genuinely different work:

* **scan** — read every row, count and total it
* **group (few)** — sixteen buckets, so almost nothing has to be held at once
* **filter** — throw away most rows early
* **group (many)** — one bucket per customer, so millions of them, and by far the heaviest

The last one is the interesting case. Grouping by customer means holding a running total per
customer, and there are a lot of customers. If anything is going to run out of memory, it is
that.

In [ ]:
QUERIES = {
    "scan": """
        SELECT count(*) AS rows, round(sum(amount),2) AS total,
               min(sale_date) AS first_day, max(sale_date) AS last_day
        FROM sales
    """,
    "group (few)": """
        SELECT channel, region, count(*) AS orders, round(sum(amount),2) AS revenue
        FROM sales GROUP BY 1,2 ORDER BY revenue DESC
    """,
    "filter": """
        SELECT count(*) AS n, round(avg(amount),2) AS avg_amount
        FROM sales WHERE amount > 250 AND channel = 'app' AND region = 'north'
    """,
    "group (many)": """
        SELECT customer_id, count(*) AS orders, round(sum(amount),2) AS spend
        FROM sales GROUP BY 1 ORDER BY spend DESC LIMIT 10
    """,
}

MAX_CUSTOMERS = 2_000_000   # a real business does not have one customer per 20 orders

def build(n, path):
    n_cust = min(MAX_CUSTOMERS, max(1000, n // 20))
    con = duckdb.connect(config={"memory_limit": f"{SCALE_MEMORY_MB}MB",
                                 "threads": str(C.CORES)})
    con.execute(f"""COPY (
      SELECT i                                                        AS sale_id,
             (hash(i*2654435761) % {n_cust})::INT                     AS customer_id,
             (hash(i*40503) % 5000)::INT                              AS product_id,
             DATE '2024-01-01' + ((hash(i*31) % 730)::INT)            AS sale_date,
             round((hash(i*41) % 30000)/100.0, 2)                     AS amount,
             ['web','store','app','partner'][1 + (hash(i*53) % 4)::INT]  AS channel,
             ['north','south','east','west'][1 + (hash(i*79) % 4)::INT]  AS region
      FROM range({n}) t(i)) TO '{path}' (FORMAT PARQUET)""")
    con.close()

rows, stopped_because = [], None

for n in LADDER:
    need_gb = n * BYTES_PER_ROW * SPILL_FACTOR / 1024**3
    free_gb = shutil.disk_usage(SCALE_DIR).free / 1024**3
    if need_gb > free_gb * DISK_HEADROOM:
        stopped_because = (f"{C.human(n)} rows needs about {need_gb:.1f} GB "
                           f"including spill, and only {free_gb:.1f} GB is free")
        break

    path = f"{SCALE_DIR}/scale.parquet".replace("\\", "/")
    t0 = time.perf_counter(); build(n, path); gen = time.perf_counter() - t0
    size_gb = os.path.getsize(path) / 1024**3

    con = duckdb.connect(config={"memory_limit": f"{SCALE_MEMORY_MB}MB",
                                 "threads": str(C.CORES),
                                 "temp_directory": SCALE_DIR})
    con.execute(f"CREATE OR REPLACE VIEW sales AS SELECT * FROM read_parquet('{path}')")

    rec = {"rows": n, "GB on disk": round(size_gb, 2), "generate (s)": round(gen, 1)}
    failed = None
    for label, sql in QUERIES.items():
        try:
            con.execute(sql).df()                      # warm-up, untimed
            t0 = time.perf_counter(); con.execute(sql).df()
            rec[label] = round(time.perf_counter() - t0, 3)
        except Exception as e:
            rec[label] = None
            failed = f"{label}: {str(e)[:100]}"
    con.close()
    os.remove(path)                                    # delete before the next size

    timed = [v for k, v in rec.items() if k in QUERIES and v]
    rec["all four (s)"] = round(sum(timed), 2) if timed else None
    if rec["all four (s)"]:
        rec["million rows/sec"] = round(n / rec["all four (s)"] / 1e6, 1)
    rows.append(rec)

    print(f"   {C.human(n):>13} rows  {size_gb:5.2f} GB  "
          f"built in {gen:5.1f}s   queried in {rec['all four (s)']}s"
          + (f"   [{failed}]" if failed else ""))

    if failed:
        stopped_because = failed
        break
    if rec["all four (s)"] and rec["all four (s)"] > STEP_TIME_BUDGET:
        stopped_because = "the step time budget was reached"
        break

df = pd.DataFrame(rows)
display(df)
if stopped_because:
    print(f"\nStopped: {stopped_because}")

## Does it scale, or does it fall over?

The chart on the left is time against rows. The one on the right is throughput — rows handled
per second.

A flat throughput line is the whole point. It means the engine is doing a constant amount of
work per row no matter how many rows there are, which is what streaming looks like. A line that
falls away at the right would mean it had started thrashing.

In [ ]:
if len(df) >= 2:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.6))

    for label in QUERIES:
        if label in df and df[label].notna().any():
            a1.plot(df["rows"], df[label], "o-", lw=2, ms=6, label=label)
    a1.set_xscale("log"); a1.set_yscale("log"); a1.minorticks_off()
    a1.set_xticks(df["rows"]); a1.set_xticklabels([C.human(int(r)) for r in df["rows"]],
                                                  fontsize=8, rotation=30, ha="right")
    a1.set_xlabel("Rows"); a1.set_ylabel("Seconds")
    a1.set_title("Time per query", fontsize=12, weight="bold")
    a1.legend(fontsize=9); a1.grid(alpha=.25)

    a2.plot(df["rows"], df["million rows/sec"], "o-", lw=2.5, ms=8, color="#3A7CA5")
    a2.set_xscale("log"); a2.minorticks_off()
    a2.set_xticks(df["rows"]); a2.set_xticklabels([C.human(int(r)) for r in df["rows"]],
                                                  fontsize=8, rotation=30, ha="right")
    a2.set_ylim(bottom=0)
    a2.set_xlabel("Rows"); a2.set_ylabel("Million rows per second")
    a2.set_title("Throughput (flat = still streaming)", fontsize=12, weight="bold")
    a2.grid(alpha=.25)

    for ax in (a1, a2):
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    plt.tight_layout(); plt.show()
else:
    print("Not enough steps completed to plot a curve.")

In [ ]:
big = df.iloc[-1]
n = int(big["rows"])
print("=" * 74)
print(f"""
  {C.human(n)} rows, {big['GB on disk']} GB of Parquet.

  Machine     : {C.plural(C.CORES, 'thread')}, {C.RAM_GB:.0f} GB RAM
  Memory cap  : {SCALE_MEMORY_MB} MB -- the same cap used at every size in the table above
  Four queries: {big['all four (s)']} seconds
  Throughput  : {big['million rows/sec']} million rows per second

  At 3,000 orders a day that table is about {n/3000/365:,.0f} years of trading.
""")
if big["GB on disk"] * 1024 > SCALE_MEMORY_MB:
    print(f"  The file is {big['GB on disk']*1024/SCALE_MEMORY_MB:.0f}x larger than the memory it was")
    print("  allowed to use, so the data was never held in memory. It was streamed off")
    print("  disk, and the memory cap never had to move.\n")
print("=" * 74)

# saved as CSV so the summary notebook, which reads results/*.json, ignores it
df.to_csv(os.path.join(C.RESULTS_DIR, "scale_curve.csv"), index=False)
shutil.rmtree(SCALE_DIR, ignore_errors=True)
print("\nScale datasets deleted.")

## What this does and does not say

**Does:** a single process on an ordinary machine handles a table far larger than its memory, at
roughly constant throughput, with no cluster, no configuration and no tuning.

**Does not:** say anything about many people querying at once, about governance or access
control, or about jobs that genuinely need more than one machine. It is one process doing one
job at a time.

The ladder stops where the disk on this machine stops, not where the engine stops. On a machine
with more room, set `BENCH_SCALE_MAX` higher and the curve keeps going.